# Session VII: Command Line Tools and Automation

Use Python to automate tasks and interact through the command line, including parsing arguments and running scripts from the terminal.

## Exercise 1: CLI Greeter and Calculator
**Objective:** Practice basic command-line parsing using a list that represents `sys.argv`.

Instead of reading real terminal input in the notebook, you will write a function that receives an argument list like:

```python
["greet", "Alice"]
["add", "3", "4"]
```

**Tasks:**
1. `run_command(argv)` should support two commands:
   - `greet <name>` → return `"Hello, <name>!"`
   - `add <x> <y>` → return the numeric sum as a string
2. Return a helpful usage message for missing or invalid input.
3. Use `try/except` to handle non-numeric values in `add`.


In [ ]:
def run_command(argv):
    """Parse a simple command list and return a response string."""
    # TODO: Handle empty input
    if not argv:
        return "Usage: greet <name> | add <x> <y>"
    # TODO: Support: greet <name>
    if argv[0] == "greet" and len(argv) == 2:
        return f"Hello, {argv[1]}!"
    # TODO: Support: add <x> <y>
    if argv[0] == "add" and len(argv) == 3:
    # TODO: Use try/except for numeric conversion
        try:
            return str(float(argv[1]) + float(argv[2]))
        except (ValueError, TypeError):
            return "Error: add requires two numeric values."
    # TODO: Return an error/usage message for bad commands
    return "Usage: greet <name> | add <x> <y>"
    pass


# --- TEST ---
print(run_command(["greet", "Alice"]))
# Expected: Hello, Alice!

print(run_command(["add", "3", "4"]))
# Expected: 7.0

print(run_command(["add", "three", "4"]))
# Expected: Error message about numeric input


---
## Exercise 2: Expense Lister with `argparse`
**Objective:** Build a real argument parser using optional flags.

You are given a list of expense records. Build a command-line style filter that can:
- show all expenses
- filter by category
- filter by a minimum amount

**Tasks:**
1. `build_parser()` → Create an `ArgumentParser` with:
   - `--category`
   - `--min` (float, default `0`)
2. `filter_expenses(expenses, category=None, min_amount=0)` → Return only matching expenses.
3. `format_expenses(expenses)` → Return a formatted multi-line string.


In [ ]:
import argparse

expenses = [
    {"date": "2026-10-01", "category": "Food", "amount": 18.50, "note": "Lunch"},
    {"date": "2026-10-02", "category": "Transport", "amount": 2.80, "note": "Metro"},
    {"date": "2026-10-02", "category": "Food", "amount": 34.20, "note": "Groceries"},
    {"date": "2026-10-03", "category": "Books", "amount": 45.00, "note": "Python book"},
]

def build_parser():
    """Return an argparse parser for expense filtering."""
    # TODO: Create ArgumentParser
    parser = argparse.ArgumentParser(description="Filter expense records")
    # TODO: Add optional --category argument
    parser.add_argument("--category", help="Filter by category")
    # TODO: Add optional --min argument (type=float, default=0)
    parser.add_argument("--min", type=float, default=0, help="Minimum amount")
    return parser
    pass

def filter_expenses(expenses, category=None, min_amount=0):
    """Return filtered expenses matching the given criteria."""
    # TODO: Loop through expenses and keep only matching rows
    filtered = []
    for expense in expenses:
        if (category is None or expense["category"] == category) and expense["amount"] >= min_amount:
            filtered.append(expense)
    return filtered
    pass

def format_expenses(expenses):
    """Return a formatted text report for a list of expenses."""
    # TODO: Handle empty list
    if not expenses:
        return "No expenses found."
    # TODO: Build a list of output lines
    lines = []
    for expense in expenses:
        lines.append(
            f"{expense.get('date', ''):<10}  {expense['category']:<12} "
            f"{expense['amount']:>8.2f}  {expense.get('note', '')}"
        )
    # TODO: Return one multi-line string
    return "\n".join(lines)
    pass


# --- TEST ---
parser = build_parser()
args = parser.parse_args(["--category", "Food", "--min", "20"])
filtered = filter_expenses(expenses, args.category, args.min)
print(format_expenses(filtered))


---
## Exercise 3: Budget Backup Script
**Objective:** Automate data backups using `pathlib`, timestamps, and file copying.

Your script should validate that the source file exists, create a `backups` folder if needed, and make a timestamped copy.

**Tasks:**
1. `make_backup_name(filename)` → Return a new filename like `budget_20261005_224700.json`
2. `backup_file(source_path, backup_dir="backups")` → Copy the source file into the backup folder.
3. Return the destination path as a string, or `None` if the source file is missing.

**First, run the setup cell below to create a sample file.**


In [ ]:
from pathlib import Path
import json

sample_budget = [
    {"category": "Food", "amount": 20.5},
    {"category": "Transport", "amount": 15.0},
]

with open("budget_data.json", "w") as file:
    json.dump(sample_budget, file, indent=2)

print("Created budget_data.json")


In [ ]:
from pathlib import Path
from datetime import datetime
import shutil

def make_backup_name(filename):
    """Return a timestamped backup filename."""
    # TODO: Use Path(filename).stem and .suffix
    path = Path(filename)
    stem = path.stem
    suffix = path.suffix
    # TODO: Add a timestamp with datetime.now().strftime(...)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return f"{stem}_{timestamp}{suffix}"
    pass

def backup_file(source_path, backup_dir="backups"):
    """Copy a file into backup_dir with a timestamped name."""
    # TODO: Convert to Path objects
    source = Path(source_path)
    backup_folder = Path(backup_dir)
    # TODO: Check whether source exists
    if not source.is_file():
        return None
    # TODO: Create backup folder if needed
    backup_folder.mkdir(parents=True, exist_ok=True)
    # TODO: Copy file with shutil.copy2
    destination = backup_folder / make_backup_name(source.name)
    shutil.copy2(source, destination)
    # TODO: Return destination path as a string
    return str(destination)
    pass


# --- TEST ---
result = backup_file("budget_data.json")
print("Backup created:", result)


---
## Exercise 4: CSV Expense Summary CLI
**Objective:** Load a CSV file, validate rows, and print category totals.

**Tasks:**
1. `load_expenses_csv(filename)` → Use `csv.DictReader` to load rows.
2. Skip malformed rows where:
   - `category` is empty
   - `amount` is missing or non-numeric
3. `summarize_by_category(expenses)` → Return a dictionary of totals.
4. `print_summary(summary)` → Print a formatted report.

**Run the setup cell below first.**


In [ ]:
import csv

rows = [
    ["date", "category", "amount", "note"],
    ["2026-10-01", "Food", "18.50", "Lunch"],
    ["2026-10-02", "Transport", "2.80", "Metro"],
    ["2026-10-02", "", "12.00", "Broken row"],
    ["2026-10-03", "Books", "forty", "Invalid amount"],
    ["2026-10-04", "Food", "21.30", "Dinner"],
]

with open("expenses_cli.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerows(rows)

print("Created expenses_cli.csv")


In [ ]:
import csv

def load_expenses_csv(filename):
    """Load valid expense rows from a CSV file."""
    # TODO: Use csv.DictReader
    expenses = []
    with open(filename, newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        for row in reader:
    # TODO: Validate category and amount
            category = (row.get("category") or "").strip()
            if not category or not row.get("amount"):
                continue
    # TODO: Convert amount to float
            try:
                amount = float(row["amount"])
    # TODO: Skip bad rows safely
            except (ValueError, TypeError):
                continue
            row["category"] = category
            row["amount"] = amount
            expenses.append(row)
    return expenses
    pass

def summarize_by_category(expenses):
    """Return {category: total_amount}."""
    # TODO: Accumulate totals in a dictionary
    totals = {}
    for expense in expenses:
        category = expense["category"]
        totals[category] = totals.get(category, 0) + expense["amount"]
    return totals
    pass

def print_summary(summary):
    """Print a formatted category summary."""
    # TODO: Handle empty summary
    if not summary:
        print("No expenses to summarize.")
        return
    # TODO: Print aligned output
    width = max(len("Category"), max(len(category) for category in summary))
    print(f"{'Category':<{width}}  {'Total':>10}")
    for category, total in sorted(summary.items()):
        print(f"{category:<{width}}  {total:>10.2f}")
    pass


# --- TEST ---
expenses = load_expenses_csv("expenses_cli.csv")
summary = summarize_by_category(expenses)
print_summary(summary)


---
## Exercise 5: Scheduling Helper
**Objective:** Generate schedule strings for cron and Windows Task Scheduler.

You are **not** actually scheduling tasks in this notebook. Instead, you will generate the correct command strings.

**Tasks:**
1. `build_cron_schedule(minute, hour, script_path)` → Return a cron line like:
   `30 18 * * * python /path/to/script.py`
2. `build_task_scheduler_command(time_str, script_path)` → Return a Windows-style command string.
3. Validate the time values and raise or handle `ValueError` for invalid input.


In [ ]:
def build_cron_schedule(minute, hour, script_path):
    """Return a cron schedule line for a daily script."""
    # TODO: Validate minute (0-59) and hour (0-23)
    if type(minute) is not int or type(hour) is not int or not (0 <= minute <= 59 and 0 <= hour <= 23):
        raise ValueError("Minute must be an integer from 0-59 and hour from 0-23.")
    # TODO: Return the cron line as a string
    import shlex
    return f"{minute} {hour} * * * python {shlex.quote(str(script_path))}"
    pass

def build_task_scheduler_command(time_str, script_path):
    """Return a Windows schtasks command string."""
    # TODO: Validate HH:MM format
    if not isinstance(time_str, str) or len(time_str) != 5 or time_str[2] != ":":
        raise ValueError("Time must use HH:MM format.")
    hours, minutes = time_str.split(":")
    if not (hours.isascii() and minutes.isascii() and hours.isdigit() and minutes.isdigit()):
        raise ValueError("Time must use HH:MM format.")
    if not (0 <= int(hours) <= 23 and 0 <= int(minutes) <= 59):
        raise ValueError("Hour must be 00-23 and minute must be 00-59.")
    # TODO: Return a command like:
    command = f'python "{script_path}"' if " " in str(script_path) else f"python {script_path}"
    escaped_command = command.replace('"', '\\"')
    return f'schtasks /Create /SC DAILY /TN BudgetBackup /TR "{escaped_command}" /ST {time_str}'
    # schtasks /Create /SC DAILY /TN BudgetBackup /TR "python script.py" /ST 18:30
    pass


# --- TEST ---
print(build_cron_schedule(30, 18, "/home/user/backup.py"))
print(build_task_scheduler_command("18:30", "C:/scripts/backup.py"))


---
## Exercise 6: Automation Suite
**Objective:** Combine parsing, loading JSON, listing expenses, backups, and summaries into one mini CLI application.

Build a tool with these subcommands:
- `list` → print all expenses
- `summary` → show totals by category
- `backup` → make a timestamped backup of the JSON file

**Tasks:**
1. `load_json_expenses(filename)` → Safely load a JSON list of expense dictionaries.
2. `build_suite_parser()` → Create an `argparse` parser with subcommands:
   - `list`
   - `summary`
   - `backup`
   Also add a common `filename` argument.
3. `run_suite(argv=None)` → Parse the arguments and perform the correct action.
4. Reuse functions from earlier exercises when possible.

**Run the setup cell below first.**


In [ ]:
import json

suite_data = [
    {"date": "2026-10-01", "category": "Food", "amount": 18.50, "note": "Lunch"},
    {"date": "2026-10-02", "category": "Transport", "amount": 2.80, "note": "Metro"},
    {"date": "2026-10-03", "category": "Food", "amount": 21.30, "note": "Dinner"},
    {"date": "2026-10-04", "category": "Books", "amount": 45.00, "note": "Python book"},
]

with open("expenses_suite.json", "w") as file:
    json.dump(suite_data, file, indent=2)

print("Created expenses_suite.json")


In [ ]:
import argparse
import json

def load_json_expenses(filename):
    """Safely load and validate a JSON expense list."""
    # TODO: Handle FileNotFoundError and json.JSONDecodeError
    try:
        with open(filename, encoding="utf-8") as file:
            data = json.load(file)
    except (OSError, json.JSONDecodeError, UnicodeError):
        return []
    # TODO: Ensure the top-level object is a list
    if not isinstance(data, list):
        return []
    for expense in data:
        if not isinstance(expense, dict):
            return []
        if not isinstance(expense.get("category"), str) or not expense["category"].strip():
            return []
        if isinstance(expense.get("amount"), bool) or not isinstance(expense.get("amount"), (int, float)):
            return []
    return data
    # TODO: Return [] on error
    return []
    pass

def build_suite_parser():
    """Return an argparse parser with subcommands."""
    # TODO: Create parser
    parser = argparse.ArgumentParser(description="Expense automation suite")
    # TODO: Add a required filename argument
    parser.add_argument("filename", help="JSON expense file")
    # TODO: Add subcommands: list, summary, backup
    subparsers = parser.add_subparsers(dest="command", required=True)
    subparsers.add_parser("list", help="List expenses")
    subparsers.add_parser("summary", help="Summarize expenses by category")
    subparsers.add_parser("backup", help="Back up the JSON file")
    return parser
    pass

def run_suite(argv=None):
    """Run the automation suite and return a message or report."""
    # TODO: Parse args
    parser = build_suite_parser()
    args = parser.parse_args(argv)
    # TODO: Load data when needed
    if args.command in ("list", "summary"):
        expenses = load_json_expenses(args.filename)
    # TODO: Route to list / summary / backup
    if args.command == "list":
        return format_expenses(expenses)
    if args.command == "summary":
        import io
        from contextlib import redirect_stdout
        output = io.StringIO()
        with redirect_stdout(output):
            print_summary(summarize_by_category(expenses))
        return output.getvalue().rstrip()
    if args.command == "backup":
        destination = backup_file(args.filename)
        if destination is None:
            return "Error: source file not found."
        return f"Backup created: {destination}"
    pass


# --- TEST ---
print(run_suite(["expenses_suite.json", "list"]))
print()
print(run_suite(["expenses_suite.json", "summary"]))
print()
print(run_suite(["expenses_suite.json", "backup"]))
